In [ ]:
# This code has the revised sexual reproduction (both selfing and random mating). For selfing, in this code, first select the 
# parents according to their relative fitness, then let them make gamets twice, plus these gamets generated to make zygotes;
# for random mating, first select the parents twice, and then make each set of parents make gamets, then plus them together to 
# make zygotes. 

# For the additive version, this is a problem of this code: even the population has extincted, the evolution continues and the 
# deleterious mutations keep accumulating. But for the multiplicative version, since none of population fitness would decline to
# 0 ('MUL', fitness = np.prod(each_locus, axis =2) always >0), which means none of the population would extinct, this problem 
# doesn't exist.

In [ ]:
from __future__ import division
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle

In [ ]:
class Populations(object):
    """Populations objects have the following attributes:
    Populations.soma - a 3D numpy array representing the number of mutant copies each individual has at each somatic locus
    Populations.germ - a 3D numpy array representing the number of mutant copies each individual has at each germline locus
    Populations.ploidy - chromosome copy number in the somatic nucleus
    Populations.mu - per nucleotide mutation rate per generation
    Populations.selcoef - selection coefficient
    Populations.current_step - number of generations (for Wright-Fisher) or time steps (for Moran) the simulated populations have undergone"""
    
    def __init__(self,nReps,nInds,nLoci,ploidy=45,mu=0.0001,selcoef=0.1, model_version ='ADD'):
        """Initiates a populations object.
        nReps - number of replicate populations
        nInds - number of individuals per population
        nLoci - number of loci per genome
        ploidy - somatic chromosome copy number
        mu - mutation rate
        selcoef - selection coefficient"""


        self.nReps = nReps
        self.nInds = nInds
        self.nLoci = nLoci
        
        self.soma = np.zeros((nReps,nInds,nLoci),dtype='int')
        self.germ = np.zeros((nReps,nInds,nLoci),dtype='int')
        self.ploidy = ploidy
        self.mu = mu
        self.selcoef = selcoef
        self.current_step = 0

        self.model_version = model_version
        

        

    def fitness(self):
        """return a numpy array containing the fitness for each individual in each population"""
        ws = (self.soma.astype('float')/self.ploidy)*self.selcoef

        if self. model_version == 'ADD':
            fitnesses = 1 - np.sum(ws,axis=2)
            
        elif self. model_version == 'MUL':
            each_locus = 1-ws
            fitnesses = np.prod(each_locus, axis =2)
            
        fitnesses[fitnesses <= 0] = 0  

        return fitnesses

            
    def relative_fitness(self):
        """return a numpy array containing each individual's relative fitness"""
        w = self.fitness()
        total_w = np.sum(w,axis=1)
        totalw = np.expand_dims(total_w,axis=1)
        relfit = w/totalw
        return relfit

                
    def Gst(self):
        """return Gst"""
        nInds = self.soma.shape[1]
        #Ht_min = 1.0/(nInds*self.ploidy)
        Ps = self.soma.astype('float')/self.ploidy
        Hs = 2*(Ps*(1-Ps))
        Pt = np.sum(self.soma, axis=1,dtype='float')/(nInds*self.ploidy)
        Ht = 2*(Pt*(1-Pt))
        Ht = np.expand_dims(Ht,axis=1)
        gst = (Ht-Hs)/Ht
        return gst

        
    def fixed_mutations(self):
        """return the number of fixed mutations withing each population"""
        nInds = self.soma.shape[1]
        fixed = np.sum(np.sum(self.soma==self.ploidy,axis=1)==nInds,axis=1)
        return fixed
            
   
    def all_wild_type_site(self):
        """return the number of fixed mutations withing each population"""
        nInds = self.soma.shape[1]
        all_wt_site = np.sum(np.sum(self.soma==0,axis=1)==nInds,axis=1)
        return all_wt_site


        
        
    #Wright_Fisher model
    def mutate_all_before(self):
        """Mutate all individuals in the population with mutation rate mu.
        Wright-Fisher model method. Use this one instead of mutate_all if you want to mutate the population before selection."""
        wt_soma = self.ploidy - self.soma
        wt_germ = 2 - self.germ
        soma_mutations = np.random.binomial(wt_soma,self.mu)
        germ_mutations = np.random.binomial(wt_germ,self.mu)
        self.soma += soma_mutations
        self.germ += germ_mutations
        return
    
    
    
    def pick_parents_all(self):
        """Randomly choose N parents to produce offspring to populate the next generation.
        Each individual's probability of being chosen is weighted by its relative fitness.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        relfit = self.relative_fitness()
        csrelfit = np.cumsum(relfit,axis=1)
        randvals = np.random.random((nReps,nInds))
        parents = map(np.searchsorted,csrelfit,randvals)
        return parents
    
    
    
    def mitosis_all(self,parents):
        """Generate mitotic offspring from all individuals selected as parents.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)
        self.soma = self.soma[rReps,parents,]
        self.germ = self.germ[rReps,parents,]
        return
    
    
    def amitosis_all(self,parents):
        """Generate amitotic offspring from all individuals selected as parents. Only one
        amitotic offspring is generated from each parent, so this method does not reflect
        the reciprocity of amitosis.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)
        good = (self.ploidy-self.soma[rReps,parents,])*2
        bad = self.soma[rReps,parents,]*2
        self.soma = np.random.hypergeometric(bad,good,self.ploidy)
        self.germ = self.germ[rReps,parents,]
        return
    
    
    def wright_fisher_step(self,asex_type='amitosis'):
        """Take populations through a single Wright-Fisher model generation"""
        self.mutate_all_before()
        parents = self.pick_parents_all()
        if asex_type == 'amitosis':
            self.amitosis_all(parents)
        elif asex_type == 'mitosis':
            self.mitosis_all(parents)
        self.current_step += 1
        return
    
    
    

    
    
    #Asexul reproduction_WF model and Moran model
    
    def step(self,model='M',asex_type='amitosis',sex_freq=None):
        """Take populations through one time step if model='M', or one generation if model='WF'"""
        if model == 'M':
            self.moran_step(asex_type)
        elif model == 'WF':
            self.wright_fisher_step(asex_type)
        return 
    



    def make_gametes(self, parents):

        # self.mutate_all_before()
        # parents = self.pick_parents_all(model_version)

        rReps = np.ones((self.nReps,self.nInds),dtype='int')*np.expand_dims(np.arange(self.nReps),axis=1)
        gametes = np.random.hypergeometric(self.germ[rReps,parents,],2-self.germ[rReps,parents,],1)

        return gametes




    def make_zygotes(self, random_mating=False):

#         self.mutate_all_before()

        if random_mating == False:
            parents = self.pick_parents_all()

            these_gametes = self.make_gametes(parents)
            those_gametes = self.make_gametes(parents)

        else: 
            first_parents = self.pick_parents_all()
            second_parents = self.pick_parents_all()

            these_gametes = self.make_gametes(first_parents)
            those_gametes = self.make_gametes(second_parents)           


        zygotes = these_gametes + those_gametes

        return zygotes




    def sex(self,random_mating=False):
        """Generate sexually produced offspring.
        
        If make_random_gametes does what it's supposed to do, then:
        
        If random_mating=True, then
        the gametes are produced from randomly chosen individuals for each population.
        If random_mating=False (default), then each zygotic genome is formed from gametes
        generated from a single parent (selfing)."""
        
        '''Sex reproduction: make a new zygotes, then it double twice by mitosis and form 4 nucleuses. Two of the 4 nucleuses
        will develop into somatic and germline genome, and another 2 will be degraded. During the development, the somatic 
        genome will increase the ploidy from 2 to 45, and the germline genome will remain diploidy.'''
        
        '''For sexual reproduction, we need to make germline genome as well as the somatic genome.'''
        
        self.mutate_all_before()
        
        zygotes = self.make_zygotes(random_mating)  # make zygotes. Here the random_mating is just an argument of self.make
                                                    # _zygotes, doesn't mean that here is the random_mating
        self.germ = zygotes  # the germline genome will be the same as the zygotes.
        

        
        hetcount = len(self.germ[self.germ==1])  # count the heterozygote loci. self.germ ==1 is just the heterozygote loci as
                                                # one of the two alleles at that locus is mutation and the other is wild type.
        hetvals = np.random.binomial(self.ploidy,0.5,hetcount) 
        self.soma[self.germ==1] = hetvals # replace the heterozygote sites (in somatic genome) with the hetvals.
        
        self.soma[self.germ==2] = self.ploidy # self.germ == 2 means the germline is homozygote of mutation. So the somatic 
                                            # developed from this germline should also be homozygote of mutation (beause there
                                            # is only mutation at the germline, so every time it only have mutation to pick to
                                            # form the somatic genome). Here replace these sites with self.ploidy (means all copies
                                            # at that locus are mutation).
        self.soma[self.germ==0] = 0  # self.germ == 0 means the germline is homozygote of wild type (0 means no mutation copy at 
                                    # that locus). Same reason as self.germ ==2, the somatic developed from this germline should 
                                    # also be homozygote of wild type. Here replace these sites with 0 (means all copies at that
                                    # locus are wild type).
        
        self.current_step +=1
        return 




    
    def get_results(self):
        """calculate stuff like mean fitness, Gst, and number of fixed mutations"""

        
        W = self.fitness()
        mW = np.nanmean(W)
        log_mW = np.log(mW)
        sdW = np.nanstd(W)
        semW = stats.sem(W,None)
        G = self.Gst()
        mG = np.nanmean(G)
        sdG = np.nanstd(G)
        semG = stats.sem(G,None)
        F = self.fixed_mutations()
        mF = np.nanmean(F)
        sdF = np.nanstd(F)
        semF = stats.sem(F,None)
        
        # if self.current_step%10 == 0: 
        #     self.total_fitness.append(W)
            
        return [mW,log_mW, sdW,semW,mG,sdG,semG,mF,sdF,semF]
    
    



    def monitor_population_mutation(self):
        '''Monitor the mutation dynamics in population level'''

        fixed = self.fixed_mutations()
        all_wt_site = self.all_wild_type_site()
        poly_site = self.nLoci-fixed-all_wt_site


        overall_mu_freq = np.sum(self.soma, axis =1)/(self.ploidy*self.nInds)

        value_all_poly_freq_pl = []
        value_all_poly_var_pl = []


        for i in range(self.nReps):
            pop_poly_freq_pl = []
            # pop_poly_var_pl = []

            for j in range(self.nLoci):
                if overall_mu_freq[i][j]!= 0 and overall_mu_freq[i][j]!= 1:
                    pop_poly_freq_pl.append(overall_mu_freq[i][j])

            if len(pop_poly_freq_pl) == 0:
                value_pop_poly_freq = 0
                value_pop_poly_var = 0
            elif len(pop_poly_freq_pl) == 1:
                value_pop_poly_freq = np.nanmean(pop_poly_freq_pl)
                value_pop_poly_var = 0
            else:
                value_pop_poly_freq = np.nanmean(pop_poly_freq_pl)
                value_pop_poly_var = np.var(pop_poly_freq_pl)


            value_all_poly_freq_pl.append(value_pop_poly_freq)
            value_all_poly_var_pl.append(value_pop_poly_var)


        meanPopFM = np.nanmean(fixed)
        stdPopFM = np.nanstd(fixed)
        sePopFM = stdPopFM/(len(fixed)**0.5)
        semPopFM = stats.sem(fixed,None)

        meanPopPM = np.nanmean(poly_site)
        stdPopPM = np.nanstd(poly_site)
        sePopPM = stdPopPM/(len(poly_site)**0.5)
        semPopPM = stats.sem(poly_site,None)     

        meanPopPF = np.nanmean(value_all_poly_freq_pl)   
        stdPopPF = np.nanstd(value_all_poly_freq_pl)
        sePopPF = stdPopPF/(len(value_all_poly_freq_pl)**0.5)
        semPopPF =  stats.sem(value_all_poly_freq_pl,None)

        meanPopPV = np.nanmean(value_all_poly_var_pl)
        stdPopPV = np.nanstd(value_all_poly_var_pl)
        sePopPV = stdPopPV/(len(value_all_poly_var_pl)**0.5)
        semPopPV = stats.sem(value_all_poly_var_pl,None)

        return [meanPopFM, stdPopFM, sePopFM, semPopFM, meanPopPM, stdPopPM, sePopPM, semPopPM, \
        meanPopPF, stdPopPF, sePopPF, semPopPF, meanPopPV, stdPopPV, sePopPV, semPopPV]



    
    def monitor_individual_mutation(self):
        '''Monitor the mutation dynamics in each site in each individual in each population'''
        # nReps, nInds, nLoci = self.soma.shape[0:3]
        # ploidy = self.ploidy
        
        all_fix_site = []
        all_poly_site = []
        
        all_poly_freq = []
        all_poly_var = []
        
        for i in range(self.nReps):
            pop_fix_site = []
            pop_poly_site = []
            
            pop_poly_freq = []
            pop_poly_var = []
            
            for j in range(self.nInds):
                ind_fix_site = 0
                ind_poly_site = 0
            
                ind_poly_freq = []
#                 ind_poly_var = 0
                
                for k in range(self.nLoci): # loci level
                    if self.soma[i][j][k] == self.ploidy:
                        ind_fix_site +=1
                    elif self.soma[i][j][k]>0 and self.soma[i][j][k]<self.ploidy:
                        ind_poly_site +=1
                        ind_poly_freq.append(self.soma[i][j][k]/self.ploidy)  # the freq in each poly site
                        
                ind_poly_freq_mean = np.nanmean(ind_poly_freq) 
                # print '1', ind_poly_freq_mean
                        
                if len(ind_poly_freq) ==0 or len(ind_poly_freq) ==1: 
                    ind_poly_var = 0
                else:
                    ind_poly_var = np.var(ind_poly_freq) # this is the variance of each individual
                    
                
                pop_fix_site.append(ind_fix_site) # no. of fixed mutation site in each individual in a population
                pop_poly_site.append(ind_poly_site) # no. of poly mutation site in each individual in a population
                
                pop_poly_freq.append(ind_poly_freq_mean) # the poly freq of each individual in a population
                pop_poly_var.append(ind_poly_var) # the variance of each individual in a population
                
            
            all_fix_site.append(np.nanmean(pop_fix_site)) # mean no. of fixed mutation site in a population
            all_poly_site.append(np.nanmean(pop_poly_site)) # mean no. of ploy mutation site in a population
            all_poly_freq.append(np.nanmean(pop_poly_freq)) # mean poly freq in a population
            all_poly_var.append(np.nanmean(pop_poly_var)) # mean poly variance in a population
            
            
        mean_fix_site = np.nanmean(all_fix_site)
        std_fix_site = np.nanstd(all_fix_site)
        se_fix_site = std_fix_site/(len(all_fix_site)**0.5)
        sem_fix_site = stats.sem(all_fix_site,None)

        mean_poly_site = np.nanmean(all_poly_site)
        std_poly_site = np.nanstd(all_poly_site)
        se_poly_site = std_poly_site/(len(all_poly_site)**0.5)
        sem_poly_site = stats.sem(all_poly_site,None)

        mean_poly_freq = np.nanmean(all_poly_freq)
        std_poly_freq = np.nanstd(all_poly_freq)
        se_poly_freq = std_poly_freq/(len(all_poly_freq)**0.5)
        sem_poly_freq = stats.sem(all_poly_freq,None)

        mean_poly_var = np.nanmean(all_poly_var)
        std_poly_var = np.nanstd(all_poly_var)
        se_poly_var = std_poly_var/(len(all_poly_var)**0.5)
        sem_poly_var = stats.sem(all_poly_var,None)
        

        return [mean_fix_site, std_fix_site, se_fix_site, sem_fix_site, mean_poly_site, std_poly_site, se_poly_site, sem_poly_site, \
        mean_poly_freq, std_poly_freq, se_poly_freq, sem_poly_freq, mean_poly_var, std_poly_var, se_poly_var, sem_poly_var]
        
        
        
    def simulate(self,stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
        """stepcount - total number of generations or time steps to run
        model - M for Moran or WF for Wright-Fisher
        asex_type - type of asexual reproduction: mitosis, amitosis (,and amitosish if using Moran model)
        sex_freq - frequency of sexual reproduction
        random_mating - see sex method documentation
        strides - how frequently to get data using the get_results method
        
        returns results as a list of lists"""
        results = [self.get_results()]
        
        ind_mutations = [self.monitor_individual_mutation()]

        pop_mutations = [self.monitor_population_mutation()]
#         results = []
        start = time.time()
    
        while self.current_step <= stepcount:
            if (sex_freq != None) and (self.current_step%sex_freq == 0):
                self.sex(random_mating)
            self.step(model,asex_type,sex_freq)
            if self.current_step%strides == 0:
                results.append(self.get_results())
                ind_mutations.append(self.monitor_individual_mutation())
                pop_mutations.append(self.monitor_population_mutation())
                
        colnames = ['meanFit','log_meanFit','sdFit','semFit','meanGst','sdGst','semGst','meanFM','sdFM','semFM']
        ind_mutation_colnames = ['meanFM', 'stdFM', 'seFM', 'semFM', 'meanPM', 'stdPM', 'sePM', 'semPM', \
        'meanPF','stdPF', 'sePF', 'semPF', 'meanPV', 'stdPV', 'sePV', 'semPV']
        pop_mutation_colnames = ['meanPopFM', 'stdPopFM', 'sePopFM', 'semPopFM', 'meanPopPM', 'stdPopPM', 'sePopPM', 'semPopPM',\
        'meanPopPF', 'stdPopPF', 'sePopPF', 'semPopPF', 'meanPopPV', 'stdPopPV', 'sePopPV', 'semPopPV']
        
        results = pd.DataFrame(np.array(results),columns=colnames)
        
        ind_mutations = pd.DataFrame(np.array(ind_mutations),columns=ind_mutation_colnames)

        pop_mutations = pd.DataFrame(np.array(pop_mutations),columns=pop_mutation_colnames)

        end = time.time()
        print 'TOTAL TIME: ',end-start
        return results, ind_mutations, pop_mutations
        

    def simulateNsave(self,outfile_1,outfile_2, outfile_3, stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount,model,asex_type,sex_freq,random_mating,strides)
        
        result = results[0]
        result.to_csv(outfile_1,index=False)
        
        ind_mutation = results[1]
        ind_mutation.to_csv(outfile_2, index=False)

        pop_mutation = results[2]
        pop_mutation.to_csv(outfile_3, index=False)
        
        return
         
        
def save_object(obj, filename):
    with open(filename, 'wb') as output:
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL) 